# PageRank Project - Ranking similarity between papers
### **Student: Aleksandra Zografska (48924A)**
### **Subject: Algorithms for Massive Datasets**


#### 0. Prep environment
- Download Spark
- Align cloudpickle version that caused errors
- Import Spark into notebook

In [1]:
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"

!pip install cloudpickle==2.2.0

import cloudpickle, shutil, os

new_cp_path = os.path.dirname(cloudpickle.__file__)
pyspark_cp_path = "/content/spark-3.1.1-bin-hadoop3.2/python/pyspark/cloudpickle"

print("Replacing:", pyspark_cp_path)
print("With:", new_cp_path)

shutil.copytree(new_cp_path, pyspark_cp_path, dirs_exist_ok=True)

print("Done!")

import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better

sc = spark.sparkContext

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 104 not upgraded.
Need to get 39.7 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libxtst6 amd64 2:1.2.3-1build4 [13.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 op


#### 1. Download and Load Dataset into memory
Using https://www.kaggle.com/datasets/Cornell-University/arxiv

In [4]:
import os
os.environ['KAGGLE_USERNAME'] = "aleksandrazografska"
os.environ['KAGGLE_KEY'] = "KGAT_190e0a2b50e5230eee516795d4851ce5"
!kaggle datasets download -d Cornell-University/arxiv


Dataset URL: https://www.kaggle.com/datasets/Cornell-University/arxiv
License(s): CC0-1.0
100% 1.58G/1.58G [00:09<00:00, 211MB/s]
100% 1.58G/1.58G [00:09<00:00, 177MB/s]


In [5]:
import zipfile

with zipfile.ZipFile("arxiv.zip", "r") as zip_ref:
    zip_ref.extractall("arxiv_data")

A regular json read didn't suffice. Session crached because of using too much ram, we need to load it by chunks.





In [6]:
import pandas as pd
chunks = pd.read_json(
    "arxiv_data/arxiv-metadata-oai-snapshot.json",
    lines=True,
    chunksize=10000
)

for chunk in chunks:
    print(chunk.head())
    break  # remove break to process full dataset

         id           submitter  \
0  704.0001      Pavel Nadolsky   
1  704.0002        Louis Theran   
2  704.0003         Hongjun Pan   
3  704.0004        David Callan   
4  704.0005  Alberto Torchinsky   

                                             authors  \
0  C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...   
1                    Ileana Streinu and Louis Theran   
2                                        Hongjun Pan   
3                                       David Callan   
4           Wael Abu-Shammala and Alberto Torchinsky   

                                               title  \
0  Calculation of prompt diphoton production cros...   
1           Sparsity-certifying Graph Decompositions   
2  The evolution of the Earth-Moon system based o...   
3  A determinant of Stirling cycle numbers counts...   
4  From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...   

                                  comments  \
0  37 pages, 15 figures; published version   
1    To appear in Graph

In [ ]:
''' As we will often print probability values,
we will modify the default setting of numpy so that
float entries in arrays are printed showing only a few decimal digits.'''
import numpy as np
np.set_printoptions(precision=3)
